# Подход 2. Расширенный датасет всех объектов недвижимости

Этот notebook запускает `51_СФЕРА_все_объекты_недвижимости_с_договорами.sql`.

В результат попадают все неудалённые объекты `nedv_ul_and_ip`. Если подтверждённая связь с договором найдена, договорные поля заполняются. Если связь не найдена, объект остаётся в результате, а договорные поля остаются пустыми.

Преимущество подхода — значительно больше объектов. Недостаток — у несвязанных объектов нельзя использовать договорную историю, страхователя и другие договорные признаки.

## 1. Библиотеки

Следующую ячейку достаточно выполнить один раз в используемом окружении.

In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]"

In [ ]:
import getpass
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 100)

## 2. Путь к проекту

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / '51_СФЕРА_все_объекты_недвижимости_с_договорами.sql').is_file():
            return folder
    raise FileNotFoundError('Не найден корень проекта с SQL 51')

PROJECT_ROOT = find_project_root(Path.cwd())
SQL_PATH = PROJECT_ROOT / '51_СФЕРА_все_объекты_недвижимости_с_договорами.sql'
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
CHECK_SCRIPT = (
    PROJECT_ROOT
    / 'МАТЕРИАЛЫ_ПРОЕКТА'
    / '05_скрипты'
    / '04_проверка_датасета'
    / 'проверить_датасет_51.py'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Корень проекта:', PROJECT_ROOT)
print('SQL:', SQL_PATH)

## 3. Подключение к Сфере

Возьми сервер, порт, базу и логин из свойств подключения DBeaver. Пароль вводится скрыто и в notebook не сохраняется.

In [ ]:
SPHERE_HOST = ''       # сервер из DBeaver
SPHERE_PORT = 5432     # порт из DBeaver
SPHERE_DATABASE = ''   # база данных из DBeaver
SPHERE_USER = ''       # логин из DBeaver

if not all([SPHERE_HOST, SPHERE_DATABASE, SPHERE_USER]):
    raise ValueError('Заполни SPHERE_HOST, SPHERE_DATABASE и SPHERE_USER')

password = getpass.getpass('Пароль от Сферы: ')
connection_url = (
    f'postgresql+psycopg://{quote_plus(SPHERE_USER)}:'
    f'{quote_plus(password)}@{SPHERE_HOST}:{SPHERE_PORT}/{SPHERE_DATABASE}'
)
engine = create_engine(connection_url, pool_pre_ping=True)

In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

## 4. Запуск SQL 51

Этот запрос читает все неудалённые объекты недвижимости и поэтому может работать дольше строгой выборки.

In [ ]:
sql = SQL_PATH.read_text(encoding='utf-8')

with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
display(expanded_df.head(3))

## 5. Быстрая проверка результата

В таблицах ниже нет исходных адресов, договоров или ИНН.

In [ ]:
required_columns = {
    'row_source', 'object_id', 'characteristics_id', 'contract_id',
    'elementary_obj_type', 'has_contract', 'has_address', 'has_target',
    'target_status'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Уникальных характеристик',
        'Уникальных договоров',
        'Строк с договором',
        'Строк с адресом',
        'Строк с target',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['characteristics_id'].nunique(dropna=True),
        expanded_df['contract_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).astype(bool).sum(),
        expanded_df['has_address'].fillna(False).astype(bool).sum(),
        expanded_df['has_target'].fillna(False).astype(bool).sum(),
        expanded_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))
display(expanded_df['target_status'].fillna('empty').value_counts(dropna=False))

## 6. Сохранение CSV

Результат сохраняется локально и не должен попадать в Git.

In [ ]:
output_path = OUTPUT_DIR / 'датасет_51_расширенный.csv'
expanded_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)

## 7. Агрегатный паспорт датасета

Отдельный скрипт повторно читает сохранённый CSV и формирует Markdown только с агрегатами.

In [ ]:
report_path = OUTPUT_DIR / 'паспорт_датасета_51.md'
subprocess.run(
    [sys.executable, str(CHECK_SCRIPT), str(output_path), '--output', str(report_path)],
    check=True,
)
print('Паспорт сохранён:', report_path)

In [ ]:
engine.dispose()
print('Подключение закрыто')